# Multi-Knapsack Optimization with Preference-Based Item Assignment


## Packages and Dependencies

In [ ]:
import pandas as pd

from ortools.linear_solver import pywraplp

## Data Loading

In [ ]:
#Bags
Bags = pd.read_csv('bags.csv', sep= ';')

#Items
Items = pd.read_csv('Items.csv', sep= ';')


In [ ]:
Items

In [ ]:
Bags

## Input Data

This section focuses on preparing and structuring the raw data into the specific format required by the optimization function.

* **weights**: A vector containing the weights of the items.

* **values**: A vector containing the importance (or value) of each item.

* **capacities**: A vector containing the capacity limits of each knapsack.

### Bags

In [ ]:
bags = list(Bags['ID'].unique())
print(bags)

In [ ]:
bags_capacity = dict(zip(Bags['ID'],Bags['Capacity (kg)']))
print(bags_capacity)

### Items

In [ ]:
items = list(Items['ID'].unique())
print(items)

In [ ]:
items_weight = dict(zip(Items['ID'],Items['Weight (kg)']))
print(items_weight)

### Item preferences per bag

In [ ]:
values={}
max_value = Items['Value (USD)'].max()

for item in items:
    ##------------------------Items------------------------##
    
    #print(item)
    
    item_ = Items.loc[Items['ID'] == item].iloc[0]
    
    #Item Temperature
    item_temperature = item_['Temperature']
    #print(item_temperature)
    
    #Item Humidity
    item_humidity = item_['Humidity']
    #print(item_humidity)
    
    #Item Value
    item_value = item_['Value (USD)']
    #print(item_value)
    
    for bag in bags:
        ##------------------------Bags------------------------##
        
        #print(bag)
        
        bag_ = Bags.loc[Bags['ID'] == bag].iloc[0]
        
        #Bag Temperature
        bag_temperature = bag_['Temperature']
        #print(bag_temperature)

        #Bag Humidity
        bag_humidity = bag_['Humidity']
        #print(bag_humidity)
        
        #----------------------------Score------------------------------#
        
        #Temperature
        score_temperature = 1 if bag_temperature == item_temperature else 0
        #print(score_temperature)
              
        #Humidity
        score_humidity = 1 if bag_humidity == item_humidity else 0
                
        score_total =   round((score_temperature + 2*score_humidity + item_value/max_value)/4 , 3 )
        
        if bag not in values:
            values[bag] = []

        values[bag].append(score_total)

## Optimization
### Indices

- $i = 1, \dots, m$: items

- $b = 1, \dots, n$: bags

### Parameters

- $w_i$: weight of item $i$

- $v_{ib}$: value of item $i$ assigned to bag $b$

- $c_b$: capacity of bag $b$

### Creating the optimization solver using the OR-Tools SCIP backend

In [ ]:
solver = pywraplp.Solver.CreateSolver("SCIP")

### Decision Variables

Binary variable:

$$
x_{i,b} =
\begin{cases}
1, & \text{if item } i \text{ will be assigned to bag } b \\
0, & \text{otherwise}
\end{cases}
$$

$$
x_{i,b} \in \{0,1\},
\quad
\forall i = 1, \dots, m,
\quad
\forall b = 1, \dots, n
$$

In [ ]:
x = {}

for item in items:
    for bag in bags:
        x[item, bag] = solver.BoolVar(f'x[{item},{bag}]')

### Restrictions

#### Capacity restriction
$$
\sum_{i=1}^{m} w_i x_{ib} \leq c_b \quad \forall b=1, \dots,n
$$

In [ ]:
for bag in bags:
    solver.Add(sum(x[item, bag] * items_weight[item] for item in items) <= bags_capacity[bag])

#### Item Restriction

Each item can be assigned to a maximum of one bag:

$$
\sum_{b=1}^{n} x_{ib} \leq 1
\quad
\forall i = 1, \dots, m
$$

In [ ]:
for item in items:
    solver.Add(sum(x[item, bag] for bag in bags) <= 1)

### Objective Function

Maximize the total value of items assigned to backpacks:

$$
\max
\sum_{i=1}^{m}
\sum_{b=1}^{n}
v_{ib} x_{ib}
$$

In [ ]:
solver.Maximize(sum(values[bag][i] * x[item, bag] for i, item in enumerate(items) for bag in bags))

### Solving the Optimization Model

In [ ]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    cont_i=0
    print("Total value:", solver.Objective().Value())
    for bag in bags:
        for i, item in enumerate(items):
            if x[item, bag].solution_value() == 1:
                print(f"Item {item} assigned to bag {bag} with value  {values[bag][i]}")
                cont_i +=1
    print(cont_i)                
else:
    print("It was not possible to find an optimal solution.")